# Support Integrity Auditor (SIA) — Full Pipeline
**Pseudo-labeling → Feature Engineering → Training → Inference → Dossiers**

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import os
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs("models", exist_ok=True)
print("Ready.")

## 1. Load Data

In [ ]:
df = pd.read_csv("test_tickets.csv")
print(f"Rows: {len(df)}  |  Columns: {list(df.columns)}")
df.head()

In [ ]:
df['Priority_Level'].value_counts().plot(kind='bar', title='Priority Distribution')
plt.tight_layout()
plt.show()

## 2. Signal Functions
Two heuristic signals that estimate real severity from raw ticket content.

In [ ]:
def text_severity(subject, description):
    text = (str(subject) + " " + str(description)).lower()
    critical = ["crash", "down", "dead", "emergency", "cannot access", "outage"]
    high     = ["error", "failed", "unable", "broken"]
    score = 0.30
    if any(w in text for w in critical):
        score = 0.90
    elif any(w in text for w in high):
        score = 0.65
    if text.count("!") >= 2:
        score = min(score + 0.10, 0.95)
    if "urgent" in text:
        score = min(score + 0.08, 0.95)
    return float(np.clip(score, 0, 1))


def metadata_severity(hours, satisfaction, category):
    res_sev = (
        0.90 if hours <= 1  else
        0.80 if hours <= 2  else
        0.65 if hours <= 6  else
        0.50 if hours <= 24 else 0.30
    )
    sat_sev = (
        0.85 if satisfaction <= 1 else
        0.65 if satisfaction <= 2 else
        0.45 if satisfaction <= 3 else 0.20
    )
    cat_risk = {
        "Technical": 0.75, "Billing": 0.70,
        "General": 0.35,   "Feature_Request": 0.25,
    }.get(category, 0.50)
    return float(np.clip(res_sev * 0.40 + sat_sev * 0.35 + cat_risk * 0.25, 0, 1))


print("Signal functions defined.")

## 3. Pseudo-Labeling
Since we have no ground-truth mismatch labels, we derive them:
1. Compute inferred severity from signals
2. Compare to the assigned priority
3. Tickets in the top 25% severity delta → labelled **mismatch (1)**

In [ ]:
PRIORITY_MAP = {"Critical": 0.95, "High": 0.70, "Medium": 0.45, "Low": 0.15}

signal_1 = np.array([
    text_severity(df.iloc[i]["Ticket_Subject"], df.iloc[i]["Ticket_Description"])
    for i in range(len(df))
])
signal_2 = np.array([
    metadata_severity(
        df.iloc[i]["Resolution_Time_Hours"],
        df.iloc[i]["Satisfaction_Score"],
        df.iloc[i]["Issue_Category"],
    )
    for i in range(len(df))
])

inferred   = signal_1 * 0.60 + signal_2 * 0.40
assigned   = df["Priority_Level"].map(PRIORITY_MAP).values
delta      = np.abs(inferred - assigned)
threshold  = np.percentile(delta, 75)
y          = (delta > threshold).astype(int)

print(f"Delta threshold (75th pct): {threshold:.3f}")
print(f"Mismatch (1): {y.sum()}  |  Consistent (0): {(y==0).sum()}")

plt.figure(figsize=(8, 3))
plt.hist(delta, bins=30, color='steelblue', edgecolor='white')
plt.axvline(threshold, color='red', linestyle='--', label=f'75th pct = {threshold:.2f}')
plt.title('Severity Delta Distribution')
plt.xlabel('|Inferred − Assigned|')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
# TF-IDF on subject + description
vectorizer   = TfidfVectorizer(max_features=250, ngram_range=(1, 2), min_df=1, max_df=0.95)
text_combined = df["Ticket_Subject"].astype(str) + " " + df["Ticket_Description"].astype(str)
text_features = vectorizer.fit_transform(text_combined).toarray()

# Scale numeric metadata
scaler          = StandardScaler()
metadata        = df[["Resolution_Time_Hours", "Satisfaction_Score"]].values
metadata_scaled = scaler.fit_transform(metadata)

# Concatenate all features
signal_features = np.column_stack([signal_1, signal_2])
X = np.hstack([text_features, metadata_scaled, signal_features])

print(f"Feature matrix: {X.shape}  (samples × features)")

## 5. Training

In [ ]:
model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X, y)
print("Training complete.")

In [ ]:
# Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='f1')
print(f"5-fold F1: {cv_scores.round(3)}")
print(f"Mean F1:   {cv_scores.mean():.3f}  ±  {cv_scores.std():.3f}")

In [ ]:
# Full-dataset evaluation
y_pred  = model.predict(X)
y_proba = model.predict_proba(X)[:, 1]

print(classification_report(y, y_pred, target_names=["Consistent", "Mismatch"]))

cm = confusion_matrix(y, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Consistent", "Mismatch"],
            yticklabels=["Consistent", "Mismatch"])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 6. Save Model

In [ ]:
joblib.dump(model,      "models/sia_model.pkl")
joblib.dump(vectorizer, "models/vectorizer.pkl")
joblib.dump(scaler,     "models/scaler.pkl")
print("Saved: models/sia_model.pkl, vectorizer.pkl, scaler.pkl")

## 7. Inference — Load & Predict

In [ ]:
# Simulate loading fresh (as app.py and predict.py do)
model_loaded      = joblib.load("models/sia_model.pkl")
vectorizer_loaded = joblib.load("models/vectorizer.pkl")
scaler_loaded     = joblib.load("models/scaler.pkl")

df_infer = pd.read_csv("test_tickets.csv")

text_f   = vectorizer_loaded.transform(
    df_infer["Ticket_Subject"].astype(str) + " " + df_infer["Ticket_Description"].astype(str)
).toarray()
meta_f   = scaler_loaded.transform(df_infer[["Resolution_Time_Hours", "Satisfaction_Score"]].values)

s1 = np.array([text_severity(df_infer.iloc[i]["Ticket_Subject"], df_infer.iloc[i]["Ticket_Description"]) for i in range(len(df_infer))])
s2 = np.array([metadata_severity(df_infer.iloc[i]["Resolution_Time_Hours"], df_infer.iloc[i]["Satisfaction_Score"], df_infer.iloc[i]["Issue_Category"]) for i in range(len(df_infer))])

X_infer   = np.hstack([text_f, meta_f, np.column_stack([s1, s2])])
proba     = model_loaded.predict_proba(X_infer)[:, 1]
preds     = (proba >= 0.50).astype(int)
inferred  = s1 * 0.60 + s2 * 0.40
assigned  = df_infer["Priority_Level"].map(PRIORITY_MAP).values

labels = []
for i in range(len(df_infer)):
    if preds[i] == 0:
        labels.append("Consistent")
    elif inferred[i] > assigned[i]:
        labels.append("Hidden Crisis")
    else:
        labels.append("False Alarm")

results = df_infer[["Ticket_Subject", "Priority_Level"]].copy()
results["Prediction"]       = labels
results["Confidence"]       = proba.round(3)
results["Inferred_Severity"] = inferred.round(3)
results["Severity_Delta"]    = np.abs(inferred - assigned).round(3)

results.head(10)

In [ ]:
results["Prediction"].value_counts().plot(kind='bar', color=['#11998e','#ee0979','#f7971e'], title='Prediction Counts')
plt.tight_layout()
plt.show()

## 8. Dossier Generation

In [ ]:
dossiers = []

for i in range(len(df_infer)):
    if preds[i] == 0:
        continue
    row = df_infer.iloc[i]
    delta_val = abs(inferred[i] - assigned[i])
    dossiers.append({
        "ticket_id":         f"TKT-{i}",
        "assigned_priority": row["Priority_Level"],
        "inferred_severity": round(float(inferred[i]), 3),
        "mismatch_type":     labels[i],
        "severity_delta":    round(float(delta_val), 3),
        "confidence":        round(float(proba[i]), 3),
        "feature_evidence": [
            {"signal": "text_severity",     "value": round(float(s1[i]), 3), "weight": "0.60"},
            {"signal": "metadata_severity", "value": round(float(s2[i]), 3), "weight": "0.40"},
        ],
        "constraint_analysis": (
            f"Text={s1[i]:.2f}, Metadata={s2[i]:.2f}. "
            f"Assigned={assigned[i]:.2f}, Inferred={inferred[i]:.2f}."
        ),
    })

with open("dossiers.json", "w") as f:
    json.dump(dossiers, f, indent=2)

print(f"{len(dossiers)} dossiers saved to dossiers.json")
print(json.dumps(dossiers[0], indent=2) if dossiers else "No mismatches found.")

## 9. Export Results

In [ ]:
results.to_csv("predictions.csv", index=False)
print("Saved predictions.csv")
results.describe()